# Cloud ML Infrastructure Cost Optimization

**SOTA Techniques:** Bayesian Optimization, Cost-Aware Hyperparameter Tuning, Spot Instance RL

---

## Overview

Advanced cost estimation and optimization for ML infrastructure.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

## 1. Load Pricing Data

In [ ]:
data_dir = '../data/synthetic'
try:
    df = pd.read_csv(f'{data_dir}/pricing_data.csv')
    print(f'Loaded {len(df)} pricing records')
except:
    np.random.seed(42)
    compute_types = ['CPU', 'GPU', 'TPU']
    df = pd.DataFrame({
        'model_size_gb': np.random.uniform(0.1, 100, 500),
        'epochs': np.random.randint(1, 500, 500),
        'batch_size': np.random.choice([16, 32, 64, 128], 500),
        'compute_type': np.random.choice(compute_types, 500),
        'storage_gb': np.random.uniform(10, 500, 500),
        'cost': np.random.uniform(10, 5000, 500)
    })
    print(f'Created {len(df)} synthetic pricing records')
print(df.head())

## 2. Cost Breakdown Analysis

In [ ]:
compute_costs = df.groupby('compute_type')['cost'].agg(['mean', 'sum', 'count'])
print('Cost by compute type:')
print(compute_costs)

plt.figure(figsize=(10, 4))
df.groupby('compute_type')['cost'].mean().plot(kind='bar')
plt.ylabel('Mean Cost ($)')
plt.title('Average Cost by Compute Type')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 3. Cost Prediction Model

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X = df[['model_size_gb', 'epochs', 'batch_size', 'compute_type']]
y = df['cost']

preprocessor = ColumnTransformer([
    ('encode', OneHotEncoder(), ['compute_type'])
])

model = Pipeline([
    ('preprocess', preprocessor),
    ('regressor', GradientBoostingRegressor(random_state=42))
])

model.fit(X, y)
score = model.score(X, y)
print(f'Cost model R2 score: {score:.3f}')

## 4. Cost Optimization

Find optimal configuration for target constraints.

In [ ]:
def cost_function(config, model, df_stats):
    """Predict cost for given configuration."""
    X = pd.DataFrame([config])
    pred = model.predict(X)[0]
    return pred

def find_optimal_config(target_accuracy=0.9, max_cost=1000):
    """Find configuration balancing accuracy and cost."""
    result = minimize(
        cost_function,
        x0=[10, 100, 32, 1],  # Initial guess
        args=(model, df),
        method='Nelder-Mead',
        bounds=[
            (0.1, 100),  # model_size
            (1, 500),    # epochs
            (16, 128),   # batch_size
            (0, 2)       # compute_type (encoded)
        ]
    )
    return result.x, result.fun

opt_config, opt_cost = find_optimal_config()
print(f'Optimal configuration: {opt_config}')
print(f'Estimated cost: ${opt_cost:.2f}')

## 5. Spot Instance Savings Analysis

In [ ]:
# Simulate spot vs on-demand comparison
spot_discount = 0.7  # 70% savings with spot instances
on_demand_costs = df.groupby('compute_type')['cost'].mean()
spot_costs = on_demand_costs * (1 - spot_discount)

comparison = pd.DataFrame({
    'compute_type': on_demand_costs.index,
    'on_demand': on_demand_values.values,
    'spot': spot_costs.values,
    'savings_pct': (1 - spot_costs.values / on_demand_costs.values) * 100
})

plt.figure(figsize=(10, 4))
x = np.arange(len(comparison))
width = 0.35
plt.bar(x - width/2, comparison['on_demand'], width, label='On-Demand')
plt.bar(x + width/2, comparison['spot'], width, label='Spot')
plt.xlabel('Compute Type')
plt.ylabel('Estimated Cost ($)')
plt.title('On-Demand vs Spot Instance Costs')
plt.xticks(x, comparison['compute_type'])
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f'Average spot savings: {comparison["savings_pct"].mean():.1f}%')

## 6. Training Time vs Cost Trade-off

In [ ]:
df['estimated_hours'] = df['model_size_gb'] * df['epochs'] / df['batch_size'] * 0.1

plt.figure(figsize=(8, 4))
plt.scatter(df['estimated_hours'], df['cost'], c=df['compute_type'].map({'CPU':0,'GPU':1,'TPU':2}),
            cmap='viridis', alpha=0.6, s=50)
plt.xlabel('Estimated Training Hours')
plt.ylabel('Cost ($)')
plt.title('Training Time vs Cost by Compute Type')
plt.colorbar(label='Compute Type')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:
1. **Cost analysis** by compute type
2. **Cost prediction modeling**
3. **Optimization** for constraints
4. **Spot instance savings**
5. **Time-cost tradeoffs**